# P2 lab: documents, queries, relevance and the inverted index

Lesson: [`concepts/P2_ir_framing.md`](../concepts/P2_ir_framing.md), Part H.1. Concept card: [02 P2](../02_RAG_MASTER_LEARNING_PLAN.md#p2).

**How this lab works (blank-file first).**
- Each task has a **task cell** (what to build), an **empty code cell** for you, and a **CHECK cell** (`# CHECK`, do not edit). Run the CHECK cell after your code. If it fails, read the message and fix your code.
- This notebook contains **no solution code**. Write every line yourself. Plain Python is enough (`re`, `collections`). The reference script in `sources/verification/P2/` is a solution: open it only after an honest attempt and a hint from the tutor (CR-04).
- Where a task says **Predict first**, write your answer in the prediction cell **before** you run anything.
- **No API calls** in this lab. Never print values from `.env`.
- Kernel: the project **`.venv` (Python 3.12)**, as in `labs/O2_kernel_check.ipynb`.

## Setup (given)
Run this cell. It defines the toy collection (10 documents, 5 queries, written for this lesson), the PDF path for Task 8, and `_fp`, a fingerprint that the CHECK cells use so the expected answers stay hidden until you have predicted.

In [ ]:
import hashlib, json
PDF = r"C:\Yash\New-Work\Book RAG\upstream\RAG_Techniques\data\Understanding_Climate_Change.pdf"

def _fp(x):
    """Fingerprint of a value. The CHECK cells compare fingerprints, not visible answers."""
    return hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()[:16]

DOCS = {
    "d01": "Burning coal, oil and gas releases carbon dioxide, the main cause of recent global warming.",
    "d02": "What is the main cause of climate change? Students often ask this question in class.",
    "d03": "Greenhouse gases trap heat in the atmosphere like a blanket around the Earth.",
    "d04": "Sea levels are rising because warmer ocean water expands and glaciers melt.",
    "d05": "Melting ice sheets in Greenland and Antarctica add water to the oceans.",
    "d06": "Forests absorb carbon dioxide from the air as trees grow.",
    "d07": "Solar panels and wind turbines generate electricity without burning fuel.",
    "d08": "Climate change news, climate change videos, climate change quiz: subscribe now!",
    "d09": "Cattle and rice paddies release methane, a powerful greenhouse gas.",
    "d10": "Heat waves are becoming longer and more frequent in many regions.",
}
QUERIES = {
    "q1": "main cause of climate change",
    "q2": "why do sea levels rise",
    "q3": "renewable energy",
    "q4": "where does methane come from",
    "q5": "is extreme heat becoming more common",
}
assert _fp(DOCS) == "1edaacca3ff916da" and _fp(QUERIES) == "acb0163a7a3906b8", "the toy data was edited; re-download the lab"
print(len(DOCS), "documents,", len(QUERIES), "queries")

## Task 1: a tokenizer for the index

Write `tokenize(text)` that returns a `list[str]`:
- lowercase the text;
- a token is a maximal run of the letters `a`–`z` or the digits `0`–`9`;
- everything else (spaces, punctuation, hyphens) separates tokens;
- keep duplicates and order; no stemming; no stopword list.

Example: `tokenize("Sea-level RISE, 2019!")` gives `["sea", "level", "rise", "2019"]`.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p2_measure.py).

In [ ]:
# CHECK Task 1 (do not edit)
assert tokenize("Sea-level RISE, 2019!") == ["sea", "level", "rise", "2019"], f"got {tokenize('Sea-level RISE, 2019!')}"
assert tokenize("") == [] and tokenize("...") == [], "text with no letters or digits gives []"
assert _fp([tokenize(DOCS[d]) for d in sorted(DOCS)]) == "043587a6e439e8a1", "some document tokenizes differently; check case and punctuation"
print("Task 1 OK:", tokenize(DOCS["d08"]))

## Task 2: build the inverted index

**Predict first.** The 10 documents have 119 tokens in total. How many **distinct** terms (the vocabulary) do you expect? Write it in the next cell before you build anything.

In [ ]:
my_prediction_vocab_size = None   # replace None with your number, BEFORE you build the index

Write `build_index(docs)`:
- input: a dict `doc_id -> text`;
- output: a dict `term -> postings`, where postings is a **sorted list of doc ids** that contain the term, each id **once**.

Then build `index = build_index(DOCS)`.

You must not loop over all documents at query time later; the index is what lets you avoid that.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p2_measure.py).

In [ ]:
# CHECK Task 2 (do not edit)
assert my_prediction_vocab_size is not None, "write your prediction first (it is part of the evidence)"
assert isinstance(index, dict) and all(isinstance(v, list) for v in index.values()), "postings must be lists"
assert all(v == sorted(set(v)) for v in index.values()), "each postings list must be sorted, with no duplicates"
assert all(d in index[t] for d in DOCS for t in tokenize(DOCS[d])), "every token of every document must point back to that document"
assert _fp(index) == "a75adb6a2aa3ee3a", "the index is not right yet: compare a few postings lists with the documents by hand"
print(f"Task 2 OK. Vocabulary: {len(index)} terms (you predicted {my_prediction_vocab_size}).")
print("postings for 'climate':", index["climate"], "| 'the':", index["the"], "| 'methane':", index["methane"])

**Look at it.** Which terms have the longest postings lists? Are they useful for finding relevant documents?

*Template:* "The longest lists belong to ____. They are (useful / not useful) because ____."

*Your answer:*

## Task 3: Boolean retrieval (AND, OR) using only the index

Write two functions. Both take `(query, index)`, tokenize the query with your `tokenize`, use **only** the postings lists, and return a **sorted list of doc ids**:
- `search_and(query, index)`: documents that contain **every** distinct query term;
- `search_or(query, index)`: documents that contain **at least one** query term.

A term that is not in the index has an empty postings list. An empty query returns `[]`.

**Predict first.** Read q1 (`"main cause of climate change"`) and the 10 documents. Which document(s) will `search_and` return for q1?

In [ ]:
my_prediction_and_q1 = None   # e.g. ["d05"]; replace None BEFORE you run your function

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p2_measure.py).

In [ ]:
# CHECK Task 3 (do not edit)
assert my_prediction_and_q1 is not None, "write your prediction first"
assert search_and("", index) == [] and search_or("", index) == [], "an empty query returns []"
assert search_and("photosynthesis", index) == [] and search_or("photosynthesis", index) == [], "an unknown term matches nothing"
AND = {q: search_and(QUERIES[q], index) for q in sorted(QUERIES)}
OR = {q: search_or(QUERIES[q], index) for q in sorted(QUERIES)}
assert _fp(AND) == "5375cb3802db14a1", f"search_and is not right yet: {AND}"
assert _fp(OR) == "946fea0d80376ae5", f"search_or is not right yet: {OR}"
for q in sorted(QUERIES):
    print(f"{q} {QUERIES[q]!r:42s} AND={AND[q]}  OR={OR[q]}")
print(f"Task 3 OK. You predicted AND(q1) = {my_prediction_and_q1}")

**Two observations, one sentence each.**

1. *q3 ("renewable energy") returns nothing, yet d07 is about renewable energy.* "This happens because ____."
2. *AND returns one document for q1. Read it.* "It is (relevant / not relevant), because ____."

## Task 4: from a set to a ranking

Boolean retrieval gives a **set**. Now produce a **ranked list** from the same index.

Write `rank_by_overlap(query, index)`:
- score of a document = the number of **distinct** query terms it contains;
- compute it by walking the postings lists of the query terms and adding 1 per hit (do not loop over `DOCS`);
- return a list of `(doc_id, score)` pairs for documents with score ≥ 1, sorted by score (high first), ties by doc id (ascending).

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p2_measure.py).

In [ ]:
# CHECK Task 4 (do not edit)
LEX = {q: rank_by_overlap(QUERIES[q], index) for q in sorted(QUERIES)}
assert all(s >= 1 for q in LEX for _, s in LEX[q]), "return only documents with score >= 1"
assert _fp(LEX) == "109e8ca0d7fce277", f"the ranking is not right yet: {LEX}"
for q in sorted(QUERIES):
    print(q, LEX[q])
print("Task 4 OK")

**Similar is not relevant.** The top document for q1 matches all five query words. Read it again.

*Template:* "The top document for q1 is ____. It scores highest because ____. It is (relevant / not relevant) because ____."

*Your answer:*

## Task 5: write relevance judgments (qrels)

This is the core evidence for P2. You judge, not the code.

**Guideline v1** (the same scale as [06 §6.4](../06_RAG_PROJECT_AND_EVALUATION_PLAN.md#64-span-level-gold-labelling-procedure)):
- **2** sufficient: the document alone answers the query;
- **1** partial: it gives part of the answer or a necessary piece;
- **0** not relevant: it does not help answer the query, even if it shares words or the topic.
- Binary rule used later: relevant = grade ≥ 1.

Fill two dicts, with keys written as `"q1:d01"`:
- `my_labels`: the grade for every pair you judge 1 or 2, plus **at least one tempting 0 per query** (a document you rejected). Pairs you leave out count as 0.
- `my_why`: one justification (5 words or more) for **every** key in `my_labels`.

Do this from the documents and the guideline only. Do not look at the lesson's Part F answer first.

In [ ]:
my_labels = {
    # "qN:dNN": 1,      (format example only: query id, colon, doc id)
}
my_why = {
    # "qN:dNN": "why this grade, in five words or more",
}

In [ ]:
# CHECK Task 5 (do not edit). It checks the format, then compares your binary labels with the reference.
_valid = {f"{q}:{d}" for q in QUERIES for d in DOCS}
assert my_labels and set(my_labels) <= _valid, f"keys must look like 'q1:d01'; bad keys: {sorted(set(my_labels) - _valid)}"
assert all(g in (0, 1, 2) for g in my_labels.values()), "grades are 0, 1 or 2"
assert set(my_why) == set(my_labels), f"every label needs a justification; missing: {sorted(set(my_labels) - set(my_why))}"
assert all(len(str(w).split()) >= 5 for w in my_why.values()), "each justification needs 5 words or more"
for q in QUERIES:
    assert any(k.startswith(q + ":") and g >= 1 for k, g in my_labels.items()), f"{q}: label at least one document 1 or 2 (or explain to the tutor why none is relevant)"
    assert any(k.startswith(q + ":") and g == 0 for k, g in my_labels.items()), f"{q}: add at least one tempting document you rejected (grade 0)"
_REF = set(["0d4c5fe487a8d014", "0eb0d519e0df2383", "20b6aaa6be42b957", "2a9abccb8288f3f1", "2b65b639b747fa82", "2ccb17d986628ff9", "307a963384a324dd", "30c556854307a89f", "31332e1ad049da56", "4053e39d16e21a52", "40faffb92e074110", "44212b73e827e0df", "58712ffff51d861d", "6c5783472bd3b59f", "6c9f8efca70d6e68", "701933a4cbc6af20", "7226dd260299457e", "73fad87f1be647d8", "747ef26178936b81", "74bacbd54df9d70e", "7635ceb50e83ee0e", "772b0ea9e8b3dbbd", "7b3fda29c63795f4", "7b5dbf6617f05899", "7d128e1040166127", "7e87dfc0699d768e", "805273f7e0dfc441", "8887bacf4f70e50c", "924a75cbf316acd9", "9fe1851b5a9e6243", "ab7ac1b402b45aca", "b3a3b64ab565d363", "b96944d3f64cbb33", "bae07ca8bf08459b", "bd9ec4d0b5b1f251", "c2b61d0959a34745", "c34a90e7a1c278a8", "ca09e8da208ad0d0", "cab89c0e362d9e3b", "ccfe54af6eab0858", "cf4932d5affd7c89", "d22d115c8c579454", "d3281a51c56f4fe6", "d503723caadf90c3", "d55cb627f161b9b4", "d9fbce20ef0ad6c8", "dc3304e2a0b018c0", "f57a9cbc9c9c9290", "fa5437b7b5ba53c8", "fd32eaed9d3d1eb4"])   # one fingerprint per (query, doc, binary label) of the reference qrels; the labels stay hidden
_disagree = [f"{q}:{d}" for q in sorted(QUERIES) for d in sorted(DOCS)
             if _fp([q, d, int(my_labels.get(f"{q}:{d}", 0) >= 1)]) not in _REF]
print(f"Task 5 OK (format). Binary agreement with the reference labels: {50 - len(_disagree)} / 50 pairs.")
print("Pairs where you differ (discuss each with the tutor; a disagreement is not automatically your error):", _disagree)

## Task 6: set quality vs ranking quality

From here on, use the reference relevant set for q1 (given below; it is revealed only now). Three systems returned these top-3 lists for q1.

Write three functions; each takes `(retrieved, relevant)`, where `retrieved` is a ranked list of doc ids and `relevant` a set:
- `set_precision`: relevant retrieved ÷ number retrieved (return `0.0` if nothing was retrieved);
- `set_recall`: relevant retrieved ÷ number relevant;
- `first_relevant_rank`: the 1-based position of the first relevant document, or `None` if there is none.

Then build `toy_results = {"A": [p, r, rank], "B": [...], "C": [...]}`, with `p` and `r` rounded to 4 decimals.

**Predict first:** do runs A and B get the same precision? The same first relevant rank?

In [ ]:
REL_Q1 = {"d01", "d03", "d09"}                       # reference qrels for q1 (grade >= 1)
RUNS = {"A": ["d02", "d01", "d08"], "B": ["d01", "d08", "d02"], "C": ["d01", "d03", "d09"]}
my_prediction_A_vs_B = None   # e.g. "same precision, different rank"; replace None BEFORE you compute

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p2_measure.py).

In [ ]:
# CHECK Task 6 (do not edit)
assert my_prediction_A_vs_B is not None, "write your prediction first"
assert set_precision([], REL_Q1) == 0.0, "nothing retrieved: precision 0.0"
assert first_relevant_rank(["d02", "d08"], REL_Q1) is None, "no relevant document: None"
assert first_relevant_rank(["d02", "d03"], REL_Q1) == 2, "ranks are 1-based"
assert set_precision(["d01", "d02", "d03", "d04"], REL_Q1) == 0.5, "precision = relevant retrieved / number retrieved"
assert set_recall(["d01"], REL_Q1) == 1 / 3, "recall = relevant retrieved / number relevant"
assert _fp(toy_results) == "9d72704a91f3c210", f"toy_results is not right yet: {toy_results}"
for n in "ABC":
    print(n, RUNS[n], "precision, recall, first relevant rank =", toy_results[n])
print("Task 6 OK. You predicted:", my_prediction_A_vs_B)

**Which question does each number answer?** One sentence each.

*Template:* "A and B have the same ____ but a different ____. Precision and recall answer '____?'. The first relevant rank answers '____?'."

*Your answer:*

## Task 7: what if k doubles? (CR-08)

Use your `rank_by_overlap` ranking for q1 and `REL_Q1`.

**Predict first:** if k goes from 3 to 6, does the set recall for q1 go up?

Then build `cr08 = [len(ranking_q1), round(recall_at_3, 4), round(recall_at_6, 4)]`, where `ranking_q1` is the list of doc ids from `rank_by_overlap(QUERIES["q1"], index)` and the recalls use its top 3 and top 6.

In [ ]:
my_prediction_k6 = None   # "up", "same" or "down"; replace None BEFORE you compute

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p2_measure.py).

In [ ]:
# CHECK Task 7 (do not edit)
assert my_prediction_k6 in ("up", "same", "down"), "write your prediction first: 'up', 'same' or 'down'"
assert _fp(cr08) == "f0320f3ce2231c5f", f"cr08 is not right yet: {cr08}"
print(f"Task 7 OK. Candidates={cr08[0]}, recall@3={cr08[1]}, recall@6={cr08[2]}; you predicted '{my_prediction_k6}'.")

**Why?** One sentence.

*Template:* "Recall does not change because the index only finds ____, so a larger k ____."

*Your answer:*

## Task 8: the same index on the climate PDF

The cell below (given) cuts the repo's climate PDF into 91 plain windows of 1,000 characters with overlap 200, as in P1. Run it.

**Predict first:** in how many of the 91 windows does the word `the` appear?

In [ ]:
import pymupdf
_full = "\n".join(p.get_text() for p in pymupdf.open(PDF))
WINDOWS = {f"w{i // 800:02d}": _full[i:i + 1000] for i in range(0, len(_full), 800)}
print(len(_full), "characters,", len(WINDOWS), "windows")
my_prediction_df_the = None   # replace None BEFORE you build the PDF index

Build `pdf_index = build_index(WINDOWS)` with **your** function from Task 2. Then build

`pdf_facts = [number of windows, vocabulary size, windows containing "the", windows containing "methane", len(search_or("methane emissions", pdf_index))]`.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p2_measure.py).

In [ ]:
# CHECK Task 8 (do not edit)
assert my_prediction_df_the is not None, "write your prediction first"
assert _fp(pdf_facts) == "552eae7370ad8bbe", f"pdf_facts is not right yet: {pdf_facts}"
print(f"Task 8 OK. {pdf_facts[1]} terms; 'the' in {pdf_facts[2]} of {pdf_facts[0]} windows (you predicted {my_prediction_df_the}); "
      f"'methane' in {pdf_facts[3]}; OR('methane emissions') returns {pdf_facts[4]} windows.")
print("OR('main cause of climate change') returns", len(search_or("main cause of climate change", pdf_index)), "windows.")

**Set or ranking?** One sentence.

*Template:* "OR on q1's words returns almost every window, so a Boolean set is ____; we need ____ instead."

*Your answer:*

## Wrap-up: evidence for the log

Answer each in one line, no notes.

1. *Inverted index in one sentence:* "An inverted index maps each ____ to ____."
2. *Similar vs relevant:* "d02 is the most ____ document for q1 but it is not ____, because ____."
3. *Your labels:* "I agreed on ____ / 50 pairs. The pair I would defend is ____, because ____."
4. *Set vs ranking:* "Runs A and B differ in ____ but not in ____."

> Tell the tutor when you are done. The tutor grades `coding_independence_0_3` (Tasks 2, 3, 4) and `debugging_0_3` (any failed CHECK you fixed yourself). On a later day the tutor will ask you to write `build_index` again in a blank cell, from memory (02 P2 evidence).